In [ ]:
# CELL -- FLOP Estimation from results.jsonl
import json
from transformers import AutoTokenizer

# ── Config (must match your original CONFIG) ──────────────────────
GUIDE_PARAMS   = 3.0e9    # Qwen 3B
SOLVER_PARAMS  = 1.5e9    # Qwen 1.5B
N_VOTES        = 5
MAX_NEW_TOKENS = 350      # your max_new_tokens setting

GUIDE_MODEL  = "Qwen/Qwen2.5-3B-Instruct"
SOLVER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizers (no GPU needed, CPU only)
guide_tok  = AutoTokenizer.from_pretrained(GUIDE_MODEL)
solver_tok = AutoTokenizer.from_pretrained(SOLVER_MODEL)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def count_tokens(tok, text):
    return len(tok.encode(text, add_special_tokens=False))

def flops(n_params, n_tokens):
    """Standard approximation: 2 × N × T"""
    return 2 * n_params * n_tokens

# ── Load results ──────────────────────────────────────────────────
with open("/content/results_ARC_CHALLENGE_LLAMA3.2-1B_900.jsonl") as f:
    records = [json.loads(l) for l in f if l.strip()]

guided_records   = [r for r in records if r["mode"] == "guided"]
baseline_records = [r for r in records if r["mode"] == "baseline"]

# ── Compute FLOPs per question ────────────────────────────────────
total_guided_flops   = 0
total_baseline_flops = 0

for r in guided_records:
    q_tokens    = count_tokens(guide_tok, r["question"])
    plan_tokens = count_tokens(guide_tok, r.get("plan", ""))

    # 1) Guide call: input = question, output = plan
    guide_flops = flops(GUIDE_PARAMS, q_tokens + plan_tokens)

    # 2) Solver x N_VOTES: input = question + plan, output = MAX_NEW_TOKENS
    solver_input_tokens = count_tokens(solver_tok, r["question"] + r.get("plan", ""))
    per_solver_call     = flops(SOLVER_PARAMS, solver_input_tokens + MAX_NEW_TOKENS)
    solver_flops        = per_solver_call * N_VOTES

    # 3) Refiner (if used): same size as one solver call
    refiner_flops = per_solver_call if r.get("refiner_used") else 0

    total_guided_flops += guide_flops + solver_flops + refiner_flops

for r in baseline_records:
    solver_input_tokens = count_tokens(solver_tok, r["question"])
    per_solver_call     = flops(SOLVER_PARAMS, solver_input_tokens + MAX_NEW_TOKENS)
    total_baseline_flops += per_solver_call * N_VOTES

# ── Report ────────────────────────────────────────────────────────
n = len(guided_records)

print(f"Questions evaluated : {n}")
print(f"\n{'':30s} {'Total TFLOPs':>15} {'Per Question GFLOPs':>20}")
print("-" * 67)
print(f"{'Guided pipeline':30s} {total_guided_flops/1e12:>15.2f} {total_guided_flops/n/1e9:>20.2f}")
print(f"{'Baseline':30s} {total_baseline_flops/1e12:>15.2f} {total_baseline_flops/n/1e9:>20.2f}")
print(f"{'Overhead (guided - baseline)':30s} {(total_guided_flops-total_baseline_flops)/1e12:>15.2f}")

g_acc = sum(r["correct"] for r in guided_records)   / n * 100
b_acc = sum(r["correct"] for r in baseline_records) / n * 100

guided_flops_per_correct   = total_guided_flops   / max(sum(r["correct"] for r in guided_records), 1)
baseline_flops_per_correct = total_baseline_flops / max(sum(r["correct"] for r in baseline_records), 1)

print(f"\n{'Guided accuracy':30s} {g_acc:.1f}%")
print(f"{'Baseline accuracy':30s} {b_acc:.1f}%")
print(f"\n{'GFLOPs per correct answer':}")
print(f"  Guided   : {guided_flops_per_correct/1e9:.2f} GFLOPs")
print(f"  Baseline : {baseline_flops_per_correct/1e9:.2f} GFLOPs")
print(f"\n→ Guided uses {total_guided_flops/total_baseline_flops:.2f}x the FLOPs of baseline")
print(f"→ But earns {g_acc/b_acc:.2f}x the accuracy")

Questions evaluated : 900

                                  Total TFLOPs  Per Question GFLOPs
-------------------------------------------------------------------
Guided pipeline                        7904.42              8782.69
Baseline                               5568.03              6186.70
Overhead (guided - baseline)           2336.39

Guided accuracy                58.4%
Baseline accuracy              49.2%

GFLOPs per correct answer
  Guided   : 15027.42 GFLOPs
  Baseline : 12568.92 GFLOPs

→ Guided uses 1.42x the FLOPs of baseline
→ But earns 1.19x the accuracy
